In [ ]:
# Install dependensi yang dibutuhkan
# !pip install torch torchaudio matplotlib scikit-learn tqdm

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")
print(f"✅ PyTorch version: {torch.__version__}")


In [ ]:
# ============================================================
# EDIT BAGIAN INI sesuai kebutuhan proyek Anda
# ============================================================
CONFIG = {
    # Data
    "input_dim": 40,          # Dimensi fitur input (e.g., 40 MFCC)
    "num_classes": 10,        # Jumlah kelas / dialek Jawa
    "max_seq_len": 300,       # Panjang sekuens maksimum (frame)

    # Training
    "batch_size": 32,
    "num_epochs": 50,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,

    # Scheduler (ReduceLROnPlateau)
    "lr_patience": 5,
    "lr_factor": 0.5,
    "lr_min": 1e-6,

    # Early Stopping
    "early_stop_patience": 10,

    # Simpan model
    "checkpoint_dir": "D:/doc/2026/FINAL_CODE/src/testTDNN",
    "best_model_path": "D:/doc/2026/FINAL_CODE/src/testTDNN",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
print("📋 Konfigurasi:")
for k, v in CONFIG.items():
    print(f"   {k:25s} = {v}")


In [ ]:
class FlexibleTDNNLayer(nn.Module):
    """
    TDNN Layer dengan context delay yang bisa dikustomisasi penuh.
    Mendukung dilation untuk menangkap dependensi jarak jauh.
    """
    def __init__(self, input_dim, output_dim, context_size, dilation=1):
        super().__init__()
        self.context_size = context_size
        self.dilation = dilation

        self.conv = nn.Conv1d(
            in_channels=input_dim,
            out_channels=output_dim,
            kernel_size=context_size,
            dilation=dilation,
            padding=0  # padding manual agar konteks simetris
        )
        self.bn = nn.BatchNorm1d(output_dim)
        self.activation = nn.ReLU()

    def forward(self, x):
        # x: [batch, features, time]
        pad = (self.context_size - 1) * self.dilation // 2
        x = torch.nn.functional.pad(x, (pad, pad))
        return self.activation(self.bn(self.conv(x)))


class StatsPooling(nn.Module):
    """
    Mean + Std Pooling — agregasi seluruh sequence menjadi vektor tetap.
    Output: [batch, features * 2]
    """
    def forward(self, x):
        mean = x.mean(dim=-1)
        std = x.std(dim=-1)
        return torch.cat([mean, std], dim=-1)


class JavaneseTDNN(nn.Module):
    """
    TDNN yang dirancang untuk karakteristik Bahasa Jawa.

    Arsitektur berlapis:
      Layer 1 — context kecil  : tangkap afiks pendek
      Layer 2 — context menengah: tangkap morfem
      Layer 3 — dilated x3     : tangkap kata panjang
      Layer 4 — dilated x6     : tangkap dependensi antar kata
    """
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.layer1 = FlexibleTDNNLayer(input_dim, 256, context_size=3, dilation=1)
        self.layer2 = FlexibleTDNNLayer(256,       512, context_size=5, dilation=1)
        self.layer3 = FlexibleTDNNLayer(512,       512, context_size=3, dilation=3)
        self.layer4 = FlexibleTDNNLayer(512,       512, context_size=3, dilation=6)

        self.stats_pool = StatsPooling()  # 512 * 2 = 1024

        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.stats_pool(x)      # → [batch, 1024]
        return self.classifier(x)   # → [batch, num_classes]


# Inisialisasi model
model = JavaneseTDNN(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"]
).to(DEVICE)

# Ringkasan parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🏗️  Total parameter    : {total_params:,}")
print(f"🔧  Trainable parameter: {trainable_params:,}")
print(model)


In [ ]:
class DummyJavaneseDataset(Dataset):
    """
    Dataset dummy untuk demo pipeline.
    GANTI dengan dataset Anda sendiri — misalnya dari file .npy, .pt, atau torchaudio.

    Contoh integrasi real dataset:
        def __init__(self, manifest_path, feature_dir):
            # baca daftar file audio + label
            self.samples = pd.read_csv(manifest_path)
            self.feature_dir = feature_dir

        def __getitem__(self, idx):
            feat = np.load(os.path.join(self.feature_dir, self.samples.iloc[idx]["feat_file"]))
            label = self.samples.iloc[idx]["label"]
            return torch.tensor(feat, dtype=torch.float32), label
    """
    def __init__(self, num_samples=1000, input_dim=40, max_seq_len=300, num_classes=10):
        self.data = [
            (
                torch.randn(input_dim, random.randint(50, max_seq_len)),
                random.randint(0, num_classes - 1)
            )
            for _ in range(num_samples)
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    """Padding batch agar semua sekuens seragam panjangnya."""
    feats, labels = zip(*batch)
    max_len = max(f.size(-1) for f in feats)
    padded = torch.zeros(len(feats), feats[0].size(0), max_len)
    for i, f in enumerate(feats):
        padded[i, :, :f.size(-1)] = f
    return padded, torch.tensor(labels, dtype=torch.long)


# Buat dataset & split train/val/test (80/10/10)
full_dataset = DummyJavaneseDataset(
    num_samples=1000,
    input_dim=CONFIG["input_dim"],
    max_seq_len=CONFIG["max_seq_len"],
    num_classes=CONFIG["num_classes"]
)

n = len(full_dataset)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val
train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"📂 Dataset   : {n} sampel total")
print(f"   Train     : {len(train_ds)}")
print(f"   Validasi  : {len(val_ds)}")
print(f"   Test      : {len(test_ds)}")
print(f"   Batches/epoch (train): {len(train_loader)}")


In [ ]:
# ── Optimizer, Loss, Scheduler ──────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=CONFIG["lr_patience"],
    factor=CONFIG["lr_factor"],
    min_lr=CONFIG["lr_min"],
    verbose=True
)


# ── Early Stopping ───────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=10, delta=1e-4):
        self.patience = patience
        self.delta = delta
        self.best_loss = float("inf")
        self.counter = 0
        self.triggered = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


early_stopping = EarlyStopping(patience=CONFIG["early_stop_patience"])


# ── Satu epoch train ─────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for feats, labels in loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


# ── Satu epoch evaluasi ──────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for feats, labels in loader:
        feats, labels = feats.to(device), labels.to(device)
        logits = model(feats)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


print("✅ Fungsi training & evaluasi siap.")


In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
best_val_loss = float("inf")

print(f"🚀 Mulai training — {CONFIG['num_epochs']} epoch maksimum\n")

for epoch in range(1, CONFIG["num_epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    # Scheduler step
    scheduler.step(val_loss)

    # Simpan best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
            "config": CONFIG,
        }, CONFIG["best_model_path"])
        saved_marker = " ✅ disimpan"
    else:
        saved_marker = ""

    print(
        f"Epoch {epoch:3d}/{CONFIG['num_epochs']} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | "
        f"LR: {current_lr:.2e}{saved_marker}"
    )

    if early_stopping.step(val_loss):
        print(f"\n⏹️  Early stopping dipicu di epoch {epoch}.")
        break

print(f"\n🏆 Best val loss: {best_val_loss:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Javanese TDNN — Training History", fontsize=14, fontweight="bold")
epochs_ran = range(1, len(history["train_loss"]) + 1)

# Loss
axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss", color="#2563EB")
axes[0].plot(epochs_ran, history["val_loss"],   label="Val Loss",   color="#DC2626")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, history["train_acc"], label="Train Acc", color="#16A34A")
axes[1].plot(epochs_ran, history["val_acc"],   label="Val Acc",   color="#D97706")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(epochs_ran, history["lr"], color="#7C3AED")
axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Epoch"); axes[2].set_yscale("log"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['checkpoint_dir']}/training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Grafik disimpan ke checkpoints/training_history.png")


In [ ]:
# Load best model
checkpoint = torch.load(CONFIG["best_model_path"], map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"📦 Model terbaik dimuat — epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}\n")

# Prediksi
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for feats, labels in test_loader:
        feats = feats.to(DEVICE)
        preds = model(feats).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Classification report
target_names = [f"Kelas_{i}" for i in range(CONFIG["num_classes"])]
print("📋 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig(f"{CONFIG['checkpoint_dir']}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Export model untuk inferensi (TorchScript) ───────────────
scripted_model = torch.jit.script(model)
script_path = f"{CONFIG['checkpoint_dir']}/javanese_tdnn_scripted.pt"
scripted_model.save(script_path)
print(f"✅ TorchScript model disimpan ke: {script_path}")


# ── Contoh inferensi satu sampel ────────────────────────────
def predict_one(model, feature_tensor, device=DEVICE):
    """
    Inferensi satu sampel.
    Args:
        feature_tensor: torch.Tensor shape [input_dim, seq_len]
    Returns:
        predicted_class (int), probabilities (np.ndarray)
    """
    model.eval()
    with torch.no_grad():
        x = feature_tensor.unsqueeze(0).to(device)  # [1, dim, seq_len]
        logits = model(x)
        probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
        pred_class = probs.argmax()
    return int(pred_class), probs


# Demo
sample_feat = torch.randn(CONFIG["input_dim"], 150)
pred_class, probs = predict_one(model, sample_feat)
print(f"\n🔍 Demo inferensi:")
print(f"   Prediksi kelas : {pred_class} ({target_names[pred_class]})")
print(f"   Confidence     : {probs[pred_class]*100:.1f}%")
print(f"   Top-3 kelas    : {np.argsort(probs)[::-1][:3]}")
